# Next step: step 12c, MPI
**What it adds:** `--mode mpi`, the same per-model tasks as `--mode local`, but the workers are MPI ranks, so one
run can span several nodes.

```bash
mpirun -n N python -m mpi4py.futures -m galactic_tidal_constraints.simr --mode mpi --models ... --output ...
```
- **Rank 0** runs the command: it reads the models, writes the catalog, hands out the models and collects the
  results, exactly as the main process does in solo and local mode. **Ranks 1 to N-1** are workers; `mpi4py.futures`
  sets them up, and they never run the command themselves. `simr.run` itself didn't change.
- **`mpi4py`** is an optional group, `uv sync --group mpi`, built from source so it uses the installed MPI (OpenMPI
  4.1.6 here and on Babbage).
- **Log lines name the rank** (`[rank 0]`), read from the variable `mpirun` sets, so `simr` never imports `mpi4py`.
- **Mistakes stop early**, with exit code 1: no `mpi4py`, one rank, or `mpirun` without `-m mpi4py.futures`.
- **A worker that crashes** usually makes `mpirun` end the whole job, so there's no `SimrRunError` as in local mode.
  The catalog keeps its last write (at most a minute old) with `"finished": null`; a rerun reuses what was saved.

## What changed
**`src/galactic_tidal_constraints/simr/__main__.py`.** A third mode. It checks the launch, then hands `simr.run` the
MPI pool's `map`:
```python
def _run_mpi(models, request, args):
    try:
        from mpi4py import MPI
        from mpi4py.futures import MPIPoolExecutor
    except ImportError:
        raise ImportError("--mode mpi needs mpi4py: uv sync --group mpi") from None
    ranks, rank = MPI.COMM_WORLD.Get_size(), MPI.COMM_WORLD.Get_rank()
    if rank != 0:      # with -m mpi4py.futures, only rank 0 runs this
        raise ValueError(f"rank {rank} is running the command: launch with {launch}")
    if ranks < 2:
        raise ValueError(f"--mode mpi needs 2 or more ranks ...")
    with MPIPoolExecutor(initializer=simr.configure_logging, initargs=(args.log_level,)) as pool:
        return simr.run(models, request, output=args.output, map=pool.map)
```
**`src/galactic_tidal_constraints/simr/simr.py`.** `configure_logging` puts the rank in place of the process name
when a launcher set one:
```python
MPI_RANK_VARIABLES = ("OMPI_COMM_WORLD_RANK", "PMI_RANK")      # OpenMPI, then MPICH / Intel MPI

rank = next((os.environ[name] for name in MPI_RANK_VARIABLES if name in os.environ), None)
log_format = LOG_FORMAT if rank is None else LOG_FORMAT.replace("%(processName)s", f"rank {rank}")
```
**`pyproject.toml`.** The `mpi` group, and `[tool.uv] no-binary-package = ["mpi4py"]` to build it from source.

**Tests:** section "Step 12c" in `tests/test_simr.py` (the rank in log lines) and `tests/test_cli.py` (a slow run
under `mpirun -n 3` matching solo; no `mpi4py`; no `mpirun`; `--workers` with mpi). **Docs:**
`docs/design/simr.md` (With MPI, The command line) and `PLAN.md` (Parallelization, In `simr`).

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np

from galactic_tidal_constraints.simr import simr

base = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                 12.97415, 135.11902, 97.54035, 305.01225])
first_approach = base.copy()
first_approach[3:6] *= -3                                     # no pass before now: fails with no_pass="error"
models = np.stack([base * (1 + 0.01 * k) for k in range(7)] + [first_approach])
work = Path(tempfile.mkdtemp(prefix="simr_12c_"))
(work / "models.txt").write_text("\n".join(",".join(str(v) for v in model) for model in models) + "\n")
(work / "config.json").write_text(json.dumps({"run": {"save": ["sim.orbit", "sim.particles"]},
                                              "simulator": {"n1": 500, "n2": 500, "no_pass": "error"}}))
files = ["--models", str(work / "models.txt"), "--config", str(work / "config.json")]
simr_command = [sys.executable, "-m", "galactic_tidal_constraints.simr", *files]
mpi_command = ["mpirun", "-n", "3", "--oversubscribe", sys.executable, "-m", "mpi4py.futures",
               "-m", "galactic_tidal_constraints.simr", *files, "--mode", "mpi"]
# Docker runs as root, which OpenMPI refuses unless told otherwise. Not needed under Apptainer or on Babbage.
mpi_env = {**os.environ, "OMPI_ALLOW_RUN_AS_ROOT": "1", "OMPI_ALLOW_RUN_AS_ROOT_CONFIRM": "1"}

## 1. A run on 3 ranks
Rank 0 runs the command and 2 ranks are workers. The 8th model fails; its error is logged by the worker that ran it:

In [ ]:
done = subprocess.run(mpi_command + ["--output", str(work / "mpi")], capture_output=True, text=True, env=mpi_env)
print(done.stderr.strip())
print("exit code:", done.returncode)

## 2. The same as one process
The same command in solo mode. Every saved array is bit-identical, and the statuses match:

In [ ]:
subprocess.run(simr_command + ["--output", str(work / "solo")], capture_output=True, text=True)
for folder in ("mpi", "solo"):
    print(f"{folder:>5}:", [row[0] for row in simr.load_catalog(work / folder)["runs"][0]["status"]["sim"]])

mpi_arrays = sorted((work / "mpi").rglob("*.npy"))
solo_arrays = sorted((work / "solo").rglob("*.npy"))
same_files = [p.relative_to(work / "mpi") for p in mpi_arrays] == [p.relative_to(work / "solo") for p in solo_arrays]
print(len(mpi_arrays), "arrays, identical:",
      same_files and all(np.array_equal(np.load(a), np.load(b)) for a, b in zip(mpi_arrays, solo_arrays)))

## 3. Launch mistakes
Each stops before anything is written, with exit code 1. Without `mpirun`, or with one rank, there are no workers.
With `mpirun` but without `-m mpi4py.futures`, every rank runs the command; the ranks above 0 stop, and `mpirun`
ends the job:

In [ ]:
mistakes = {
    "no mpirun": simr_command + ["--mode", "mpi"],
    "1 rank": ["mpirun", "-n", "1"] + mpi_command[4:],
    "no -m mpi4py.futures": mpi_command[:4] + simr_command + ["--mode", "mpi"],
}
for name, command in mistakes.items():
    done = subprocess.run(command + ["--output", str(work / "mistake")], capture_output=True, text=True, env=mpi_env)
    first_error = next((line for line in done.stderr.splitlines() if "ERROR" in line), done.stderr.strip()[:300])
    print(f"{name}: exit {done.returncode}\n    {first_error}\n")
print("anything written:", (work / "mistake").exists())